# PrivBERT on OPP-115: hierarchical fine-grained classification

## Goal

Run a leakage-resistant OPP-115 workflow with one deterministic whole-policy
split:

- **65 policies for training** and **50 policies for held-out evaluation**
- Stage 1: optional domain-adaptive masked-language-model (MLM) pretraining
- Stage 2: a shared PrivBERT/RoBERTa encoder with **10 category outputs** and **20
  category-scoped, loss-masked attribute heads**
- Attribute-value vocabularies are frozen from the 65 training policies only
  (138 fine-grained values in this release)
- Primary result: **hierarchical macro-F1**, combining the 10 category labels
  and category-gated fine-grained values

Full GPU training is the default. Select `MODE = "smoke"` only for a one-step
wiring check; smoke metrics are not research results.

Full mode uses the pinned `mukund/privbert` checkpoint. PrivBERT was already
pretrained on roughly one million privacy policies, so additional OPP-115 MLM
training is available as an experiment but is disabled by default.
The notebook reports whether the held-out hierarchical macro-F1 reaches the
aspirational 0.96 target; it never substitutes that target for an observed score.

## Setup

Upload `bert_opp115_colab_workspace.zip` or place it at the configured Google
Drive path. The archive contains the runnable sources and the canonical 115
OPP-115 annotation CSVs and sanitized policy files. The setup verifies the ZIP
manifest and every payload hash, rejects unsafe archive members, extracts to
fast `/content` storage, and keeps checkpoints and result files in Drive.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive, files as colab_files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED_PROJECT_FILES = (
    "bert_opp115_pretraining.py",
    "bert_opp115_classification.py",
    "opp115_hierarchy.py",
    "opp115_policy_split.py",
    "opp115_csv_loader.py",
    "ml_layer_embeddings.py",
    "evaluation.py",
    "config.py",
    "requirements_colab_bert_opp115.txt",
)

WORKSPACE_ZIP = "/content/drive/MyDrive/BERT_OPP115_Colab/bert_opp115_colab_workspace.zip"  #@param {type:"string"}
UPLOAD_WORKSPACE_IF_MISSING = True  #@param {type:"boolean"}
DRIVE_RUNS_ROOT = "/content/drive/MyDrive/BERT_OPP115_Colab/runs"  #@param {type:"string"}
DRIVE_PROJECT_DIR = ""  #@param {type:"string"}


def missing_project_files(candidate):
    return [name for name in REQUIRED_PROJECT_FILES if not (candidate / name).is_file()]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract_workspace(archive_path):
    archive_path = archive_path.expanduser().resolve()
    archive_digest = sha256_file(archive_path)
    target = Path("/content") / f"bert_opp115_{archive_digest[:12]}"
    marker = target / ".workspace_zip_state.json"
    if marker.is_file() and not missing_project_files(target):
        state = json.loads(marker.read_text(encoding="utf-8"))
        if state.get("archive_sha256") == archive_digest:
            print("Reusing verified local workspace:", target)
            return target, archive_digest

    if target.exists():
        target = Path("/content") / (
            f"bert_opp115_{archive_digest[:12]}_"
            + datetime.now().strftime("%H%M%S_%f")
        )
    staging = target.with_name(target.name + ".extracting")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)

    with zipfile.ZipFile(archive_path, "r") as archive:
        members = archive.infolist()
        if len(members) > 5000:
            raise ValueError(f"Workspace ZIP has too many members: {len(members)}")
        if sum(member.file_size for member in members) > 2 * 1024**3:
            raise ValueError("Workspace ZIP exceeds the 2 GiB extraction limit")
        staging_root = staging.resolve()
        for member in members:
            member_path = PurePosixPath(member.filename)
            unix_mode = member.external_attr >> 16
            if (
                member_path.is_absolute()
                or ".." in member_path.parts
                or (unix_mode & 0o170000) == 0o120000
            ):
                raise ValueError(f"Unsafe workspace ZIP member: {member.filename!r}")
            destination = (staging / Path(*member_path.parts)).resolve()
            try:
                destination.relative_to(staging_root)
            except ValueError as exc:
                raise ValueError(
                    f"Workspace ZIP member escapes extraction root: {member.filename!r}"
                ) from exc
        archive.extractall(staging)

    manifest_path = staging / "workspace_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("Workspace ZIP is missing workspace_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    records = manifest.get("files")
    if not isinstance(records, list) or not records:
        raise ValueError("Workspace manifest has no payload records")
    for record in records:
        relative_path = PurePosixPath(record["path"])
        if relative_path.is_absolute() or ".." in relative_path.parts:
            raise ValueError(f"Unsafe manifest path: {record['path']!r}")
        file_path = staging / Path(*relative_path.parts)
        if (
            not file_path.is_file()
            or file_path.stat().st_size != record["bytes"]
            or sha256_file(file_path) != record["sha256"]
        ):
            raise ValueError(f"Workspace manifest verification failed: {record['path']}")

    annotation_count = len(list((staging / "data/opp115/annotations").glob("*.csv")))
    policy_count = len(list((staging / "data/opp115/sanitized_policies").glob("*.html")))
    missing = missing_project_files(staging)
    if missing or annotation_count != 115 or policy_count != 115:
        raise ValueError(
            "Extracted workspace is incomplete: "
            f"missing={missing}, annotations={annotation_count}, policies={policy_count}"
        )
    (staging / ".workspace_zip_state.json").write_text(
        json.dumps(
            {
                "archive": str(archive_path),
                "archive_sha256": archive_digest,
                "annotation_csv_files": annotation_count,
                "sanitized_policy_files": policy_count,
            },
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    staging.rename(target)
    print("Extracted verified workspace to:", target)
    return target, archive_digest


WORKSPACE_SHA256 = None
WORKSPACE_MANIFEST = None
if IN_COLAB:
    drive.mount("/content/drive")
    workspace_zip_path = Path(WORKSPACE_ZIP)
    if not workspace_zip_path.is_file() and UPLOAD_WORKSPACE_IF_MISSING:
        print("Select bert_opp115_colab_workspace.zip from your computer.")
        uploaded = colab_files.upload()
        zip_uploads = [
            (name, content)
            for name, content in uploaded.items()
            if name.casefold().endswith(".zip")
        ]
        if len(zip_uploads) != 1:
            raise ValueError("Upload exactly one BERT workspace ZIP")
        uploaded_name, uploaded_content = zip_uploads[0]
        workspace_zip_path = Path("/content") / Path(uploaded_name).name
        workspace_zip_path.write_bytes(uploaded_content)

    if workspace_zip_path.is_file():
        PROJECT_DIR, WORKSPACE_SHA256 = safe_extract_workspace(workspace_zip_path)
    elif DRIVE_PROJECT_DIR.strip():
        PROJECT_DIR = Path(DRIVE_PROJECT_DIR).expanduser().resolve()
    else:
        raise FileNotFoundError(
            "Workspace ZIP not found. Set WORKSPACE_ZIP, upload it when prompted, "
            "or set DRIVE_PROJECT_DIR to a complete project folder."
        )
    if missing_project_files(PROJECT_DIR):
        raise FileNotFoundError(
            f"Project folder is incomplete: {missing_project_files(PROJECT_DIR)}"
        )
    manifest_path = PROJECT_DIR / "workspace_manifest.json"
    if manifest_path.is_file():
        WORKSPACE_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
    PYTHON_EXECUTABLE = Path(sys.executable)
else:
    PROJECT_DIR = Path.cwd().resolve()
    missing = missing_project_files(PROJECT_DIR)
    if missing:
        raise FileNotFoundError(f"Local project folder {PROJECT_DIR} is missing: {missing}")
    local_python_override = os.environ.get("BERT_OPP115_PYTHON_OVERRIDE", "").strip()
    PYTHON_EXECUTABLE = Path(
        local_python_override or sys.executable
    ).expanduser().resolve()
    if not PYTHON_EXECUTABLE.is_file():
        raise FileNotFoundError(
            f"Local Python executable does not exist: {PYTHON_EXECUTABLE}"
        )

print("Project:", PROJECT_DIR)
print("Python used for model stages:", PYTHON_EXECUTABLE)
if IN_COLAB:
    print("Workspace SHA-256:", WORKSPACE_SHA256)
    print("Persistent Drive runs root:", DRIVE_RUNS_ROOT)

In [ ]:
if IN_COLAB:
    subprocess.run(
        [
            str(PYTHON_EXECUTABLE), "-m", "pip", "install", "-q",
            "-r", str(PROJECT_DIR / "requirements_colab_bert_opp115.txt"),
        ],
        check=True,
    )
else:
    print("Local execution: using the selected Python environment.")

## Steps

### 1. Resolve the data and freeze the training-only hierarchy

In [ ]:
OPP115_OVERRIDE = ""  #@param {type:"string"}
candidates = [
    Path(OPP115_OVERRIDE).expanduser() if OPP115_OVERRIDE else None,
    PROJECT_DIR / "data" / "OPP-115_v1_0" / "OPP-115",
    PROJECT_DIR / "data" / "opp115",
]
if IN_COLAB:
    candidates.extend(
        [
            Path("/content/drive/MyDrive/opp115"),
            Path("/content/drive/MyDrive/data/opp115"),
            Path("/content/drive/MyDrive/Thesis/data/opp115"),
        ]
    )
OPP115_DIR = next(
    (
        candidate.resolve()
        for candidate in candidates
        if candidate is not None
        and (candidate / "annotations").is_dir()
        and (candidate / "sanitized_policies").is_dir()
    ),
    None,
)
if OPP115_DIR is None:
    raise FileNotFoundError(
        "Could not find OPP-115. Set OPP115_OVERRIDE to a folder containing "
        "annotations/ and sanitized_policies/."
    )

annotations_dir = OPP115_DIR / "annotations"
policies_dir = OPP115_DIR / "sanitized_policies"
annotation_policy_ids = sorted(path.stem for path in annotations_dir.glob("*.csv"))
sanitized_policy_ids = sorted(path.stem for path in policies_dir.glob("*.html"))
if len(annotation_policy_ids) != 115 or len(sanitized_policy_ids) != 115:
    raise ValueError(
        "Canonical OPP-115 must contain 115 annotation CSVs and 115 sanitized policies"
    )
if annotation_policy_ids != sanitized_policy_ids:
    raise ValueError("Annotation and sanitized-policy identifiers do not match")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from opp115_hierarchy import (
    ATTRIBUTE_HEADS,
    HIGH_LEVEL_CATEGORIES,
    derive_hierarchy_schema,
    fixed_policy_split,
)

TRAIN_POLICY_IDS, EVAL_POLICY_IDS = fixed_policy_split(
    annotations_dir, train_count=65, eval_count=50
)
HIERARCHY_SCHEMA = derive_hierarchy_schema(annotations_dir, TRAIN_POLICY_IDS)
EXPECTED_HIERARCHY_SCHEMA = HIERARCHY_SCHEMA.to_manifest()
EXPECTED_HIERARCHY_SCHEMA_SHA256 = (
    "90627a25efe47a217e03a2f09431b3256bc254867cd8f9aee675a1eafbb93b9b"
)
EXPECTED_CATEGORY_COUNT = 10
EXPECTED_ATTRIBUTE_HEAD_COUNT = 20
EXPECTED_FINE_GRAINED_LABEL_COUNT = 138

assert len(TRAIN_POLICY_IDS) == 65
assert len(EVAL_POLICY_IDS) == 50
assert not (set(TRAIN_POLICY_IDS) & set(EVAL_POLICY_IDS))
assert tuple(TRAIN_POLICY_IDS) == HIERARCHY_SCHEMA.training_policy_ids
assert len(HIGH_LEVEL_CATEGORIES) == EXPECTED_CATEGORY_COUNT
assert len(ATTRIBUTE_HEADS) == EXPECTED_ATTRIBUTE_HEAD_COUNT
assert HIERARCHY_SCHEMA.flattened_value_count == EXPECTED_FINE_GRAINED_LABEL_COUNT
assert HIERARCHY_SCHEMA.schema_sha256 == EXPECTED_HIERARCHY_SCHEMA_SHA256

if WORKSPACE_MANIFEST is not None:
    if WORKSPACE_MANIFEST.get("hierarchy_schema_sha256") != EXPECTED_HIERARCHY_SCHEMA_SHA256:
        raise ValueError("Workspace hierarchy schema hash does not match its data")
    if WORKSPACE_MANIFEST.get("fine_grained_label_count") != EXPECTED_FINE_GRAINED_LABEL_COUNT:
        raise ValueError("Workspace fine-grained vocabulary width is stale")

print("OPP-115 root:", OPP115_DIR)
print("Whole-policy split:", len(TRAIN_POLICY_IDS), "training /", len(EVAL_POLICY_IDS), "evaluation")
print(
    "Hierarchy:",
    len(HIGH_LEVEL_CATEGORIES), "categories +",
    len(ATTRIBUTE_HEADS), "attribute heads /",
    HIERARCHY_SCHEMA.flattened_value_count, "training-derived values",
)
print("Hierarchy schema SHA-256:", HIERARCHY_SCHEMA.schema_sha256)

### 2. Confirm training mode (full by default)

In [ ]:
MODE = "full"  #@param ["full", "smoke"]
RUN_PRETRAINING = False  #@param {type:"boolean"}
REQUIRE_GPU = True  #@param {type:"boolean"}
MODEL_NAME = "mukund/privbert"
MODEL_REVISION = "48228b4661fa8252bdb39ca44a4d9758f6b37f88"
TARGET_HIERARCHICAL_MACRO_F1 = 0.96
RUN_NAME = "opp115_privbert_hierarchy_v1"  #@param {type:"string"}

local_mode_override = os.environ.get("BERT_OPP115_MODE_OVERRIDE", "").strip().casefold()
if not IN_COLAB and local_mode_override:
    if local_mode_override not in {"smoke", "full"}:
        raise ValueError("BERT_OPP115_MODE_OVERRIDE must be 'smoke' or 'full'")
    MODE = local_mode_override
    RUN_NAME = f"{RUN_NAME}_{MODE}_local_validation"
if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be 'smoke' or 'full'")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,79}", RUN_NAME):
    raise ValueError("RUN_NAME must be 1-80 letters, digits, dots, underscores, or hyphens")

import torch

cuda_available = torch.cuda.is_available()
if (MODE == "full" or REQUIRE_GPU) and IN_COLAB and not cuda_available:
    raise RuntimeError(
        "A CUDA GPU is required. In Colab choose Runtime > Change runtime type > "
        "Hardware accelerator > GPU, reconnect, then run all cells again."
    )
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    probe = torch.ones((32, 32), device="cuda")
    _ = probe @ probe
    torch.cuda.synchronize()
    del probe
    print(f"GPU ready: {gpu_name} ({gpu_memory_gib:.1f} GiB)")

TRAINING_DEVICE = "cuda" if cuda_available else "cpu"
BASE_MODEL_NAME = (
    "hf-internal-testing/tiny-random-roberta"
    if MODE == "smoke"
    else MODEL_NAME
)
BASE_MODEL_REVISION = (
    "73def02fc9f13169a1ce21ad4602aae38d7cbd5a"
    if MODE == "smoke"
    else MODEL_REVISION
)

RUNS_BASE_DIR = Path(DRIVE_RUNS_ROOT).expanduser() if IN_COLAB else PROJECT_DIR / "runs"
RUN_ROOT = RUNS_BASE_DIR / RUN_NAME
RESULTS_DIR = RUN_ROOT / "results"
PRETRAIN_OUTPUT = RUN_ROOT / "privbert-opp115-dapt"
CLASSIFIER_OUTPUT = RUN_ROOT / "privbert-opp115-hierarchy-classifier"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_CONFIGURATION = {
    "schema_version": 5,
    "workspace_sha256": WORKSPACE_SHA256,
    "mode": MODE,
    "run_pretraining": RUN_PRETRAINING,
    "base_model": BASE_MODEL_NAME,
    "base_model_revision": BASE_MODEL_REVISION,
    "expected_model_type": "roberta",
    "mlm_objective": "continued domain-adaptive MLM without NSP",
    "train_policies": 65,
    "eval_policies": 50,
    "train_policy_ids": list(TRAIN_POLICY_IDS),
    "eval_policy_ids": list(EVAL_POLICY_IDS),
    "classification_threshold": 0.5,
    "target_hierarchical_macro_f1": TARGET_HIERARCHICAL_MACRO_F1,
    "pooling_strategy": "roberta_first_token_dense_tanh",
    "classifier_dropout": 0.15,
    "checkpoint_serialization": "safetensors",
    "positive_weight_power": 0.5,
    "max_positive_weight": 10.0,
    "classification_task": "hierarchical_fine_grained_segment_classification",
    "category_count": EXPECTED_CATEGORY_COUNT,
    "attribute_head_count": EXPECTED_ATTRIBUTE_HEAD_COUNT,
    "fine_grained_label_count": EXPECTED_FINE_GRAINED_LABEL_COUNT,
    "hierarchy_schema_sha256": EXPECTED_HIERARCHY_SCHEMA_SHA256,
    "hierarchy_schema": EXPECTED_HIERARCHY_SCHEMA,
}
run_configuration_path = RUN_ROOT / "run_configuration.json"
if run_configuration_path.is_file():
    saved_configuration = json.loads(run_configuration_path.read_text(encoding="utf-8"))
    if saved_configuration != RUN_CONFIGURATION:
        raise RuntimeError(
            f"Run {RUN_NAME!r} already exists with different settings. "
            "Choose a new RUN_NAME or restore the original settings."
        )
else:
    run_configuration_path.write_text(
        json.dumps(RUN_CONFIGURATION, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


def model_directory_valid(path):
    config_path = path / "config.json"
    required = (
        config_path.is_file()
        and any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))
        and (
            (path / "tokenizer.json").is_file()
            or (path / "vocab.txt").is_file()
            or ((path / "vocab.json").is_file() and (path / "merges.txt").is_file())
        )
    )
    if not required:
        return False
    try:
        return json.loads(config_path.read_text(encoding="utf-8")).get("model_type") == "roberta"
    except (OSError, TypeError, ValueError):
        return False


def classifier_directory_valid(path):
    required = (
        path / "classifier_config.json",
        path / "hierarchy_schema.json",
        path / "hierarchy_heads.safetensors",
        path / "encoder" / "config.json",
        path / "bert_opp115_hierarchy_classification_results.json",
    )
    if not all(item.is_file() for item in required):
        return False
    if not any((path / "encoder" / name).is_file() for name in ("model.safetensors", "pytorch_model.bin")):
        return False
    if not (
        (path / "tokenizer.json").is_file()
        or (path / "vocab.txt").is_file()
        or ((path / "vocab.json").is_file() and (path / "merges.txt").is_file())
    ):
        return False
    try:
        metadata = json.loads(required[0].read_text(encoding="utf-8"))
        schema = json.loads(required[1].read_text(encoding="utf-8"))
        encoder_config = json.loads(required[3].read_text(encoding="utf-8"))
        summary = json.loads(required[4].read_text(encoding="utf-8"))
        return (
            metadata.get("model_type") == "roberta"
            and metadata.get("task") == "opp115_hierarchical_fine_grained_classification"
            and metadata.get("category_labels") == list(HIGH_LEVEL_CATEGORIES)
            and metadata.get("attribute_heads") == list(ATTRIBUTE_HEADS)
            and metadata.get("hierarchy_schema", {}).get("schema_sha256") == EXPECTED_HIERARCHY_SCHEMA_SHA256
            and schema.get("schema_sha256") == EXPECTED_HIERARCHY_SCHEMA_SHA256
            and math.isclose(float(metadata.get("threshold", float("nan"))), 0.5)
            and metadata.get("pooling_strategy") == "roberta_first_token_dense_tanh"
            and metadata.get("checkpoint_serialization") == "safetensors"
            and encoder_config.get("model_type") == "roberta"
            and summary.get("results", {}).get("primary_metric") == "hierarchical_macro_f1"
        )
    except (OSError, TypeError, ValueError):
        return False


def assert_exact_policy_split(summary):
    assert summary["train_policy_ids"] == list(TRAIN_POLICY_IDS)
    assert summary["eval_policy_ids"] == list(EVAL_POLICY_IDS)
    assert not (set(summary["train_policy_ids"]) & set(summary["eval_policy_ids"]))


def stage_is_complete(
    marker_path,
    expected_specification,
    summary_path,
    model_path=None,
    summary_validator=None,
    model_validator=model_directory_valid,
):
    if not marker_path.is_file() or not summary_path.is_file():
        return False
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if marker.get("specification") != expected_specification:
            return False
        if summary_validator is not None:
            summary_validator(summary)
    except (AssertionError, KeyError, OSError, TypeError, ValueError):
        return False
    return model_path is None or model_validator(model_path)


def write_completion_marker(marker_path, specification, summary_path, model_path=None):
    payload = {
        "specification": specification,
        "summary_file": str(summary_path),
        "model_directory": str(model_path) if model_path is not None else None,
    }
    temporary = marker_path.with_suffix(marker_path.suffix + ".incomplete")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    temporary.replace(marker_path)

print("Mode:", MODE)
print("Run directory:", RUN_ROOT)
print("Training device:", TRAINING_DEVICE)

### 3. Optional continued MLM pretraining on the 65 training policies

In [ ]:
pretraining_summary_path = RESULTS_DIR / f"colab_bert_pretraining_{MODE}.json"
pretraining_marker_path = RUN_ROOT / "bert_pretraining.complete.json"
pretraining_specification = {
    **RUN_CONFIGURATION,
    "stage": "masked_language_model_pretraining",
    "block_size": 64 if MODE == "smoke" else 384,
    "batch_size": 1 if MODE == "smoke" else 4,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "max_steps": 1 if MODE == "smoke" else 0,
    "epochs": 2,
    "skip_evaluation": True,
    "device": TRAINING_DEVICE,
}
PRETRAINING_READY = False
if RUN_PRETRAINING:
    if stage_is_complete(
        pretraining_marker_path,
        pretraining_specification,
        pretraining_summary_path,
        PRETRAIN_OUTPUT,
    ):
        print("Reusing completed pretraining stage:", PRETRAIN_OUTPUT)
        pretraining_summary = json.loads(pretraining_summary_path.read_text(encoding="utf-8"))
        assert_exact_policy_split(pretraining_summary)
        assert pretraining_summary["evaluated_eval_blocks"] == 0
        PRETRAINING_READY = True
    else:
        pretraining_command = [
            str(PYTHON_EXECUTABLE),
            str(PROJECT_DIR / "bert_opp115_pretraining.py"),
            "--data-dir", str(OPP115_DIR),
            "--model-name", BASE_MODEL_NAME,
            "--train-policies", "65",
            "--eval-policies", "50",
            "--skip-evaluation",
            "--summary-file", str(pretraining_summary_path),
            "--output-dir", str(PRETRAIN_OUTPUT),
            "--overwrite-output-dir",
            "--device", TRAINING_DEVICE,
            "--model-revision", BASE_MODEL_REVISION,
        ]
        if MODE == "smoke":
            pretraining_command += [
                "--block-size", "64", "--batch-size", "1", "--max-steps", "1"
            ]
        else:
            pretraining_command += [
                "--block-size", "384",
                "--batch-size", "4",
                "--gradient-accumulation-steps", "2",
                "--learning-rate", "2e-5",
                "--epochs", "2",
            ]
        subprocess.run(pretraining_command, cwd=PROJECT_DIR, check=True)
        if not pretraining_summary_path.is_file() or not model_directory_valid(PRETRAIN_OUTPUT):
            raise RuntimeError("Pretraining finished without complete saved artifacts")
        pretraining_summary = json.loads(pretraining_summary_path.read_text(encoding="utf-8"))
        assert_exact_policy_split(pretraining_summary)
        assert pretraining_summary["evaluated_eval_blocks"] == 0
        write_completion_marker(
            pretraining_marker_path,
            pretraining_specification,
            pretraining_summary_path,
            PRETRAIN_OUTPUT,
        )
        PRETRAINING_READY = True
else:
    pretraining_summary = None
    print("Continued pretraining skipped; classification starts from pinned PrivBERT.")

### 4. Fine-tune the hierarchical classifier

The parent task predicts any applicable subset of the 10 OPP-115 categories.
Each of the 20 attribute heads reads the shared text representation and is
trained only on segments whose gold parent category makes that head applicable.
During evaluation, attribute predictions
are also gated by predicted parent categories. The attribute vocabulary and
its 138-value width were frozen before any held-out annotation values were
read.

In [ ]:
classifier_checkpoint = str(PRETRAIN_OUTPUT) if PRETRAINING_READY else BASE_MODEL_NAME
classification_summary_path = RESULTS_DIR / f"colab_bert_hierarchy_classification_{MODE}.json"
classification_marker_path = RUN_ROOT / "bert_hierarchy_classification.complete.json"
classification_specification = {
    **RUN_CONFIGURATION,
    "stage": "hierarchical_fine_grained_segment_classification",
    "model_checkpoint": classifier_checkpoint,
    "max_length": 64 if MODE == "smoke" else 384,
    "batch_size": 4 if MODE == "smoke" else 8,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2.5e-5,
    "max_steps": 1 if MODE == "smoke" else 0,
    "epochs": 5,
    "attribute_loss_weight": 1.0,
    "classifier_dropout": 0.15,
    "positive_weight_power": 0.5,
    "max_positive_weight": 10.0,
    "target_hierarchical_macro_f1": TARGET_HIERARCHICAL_MACRO_F1,
    "pooling_strategy": "roberta_first_token_dense_tanh",
    "device": TRAINING_DEVICE,
}


def assert_unit_interval(value, field_name, allow_none=False):
    if value is None and allow_none:
        return
    if not isinstance(value, (int, float)) or not math.isfinite(float(value)):
        raise ValueError(f"{field_name} must be a finite number")
    if not 0.0 <= float(value) <= 1.0:
        raise ValueError(f"{field_name} must be between 0 and 1")


def validate_classification_summary(summary, expected_model_name):
    if summary.get("corpus") != "OPP-115":
        raise ValueError("Classification summary has the wrong corpus")
    if summary.get("task") != "hierarchical fine-grained segment classification":
        raise ValueError("Classification summary has the wrong task")
    if summary.get("category_count") != EXPECTED_CATEGORY_COUNT:
        raise ValueError("Classification summary does not contain 10 categories")
    if summary.get("attribute_head_count") != EXPECTED_ATTRIBUTE_HEAD_COUNT:
        raise ValueError("Classification summary does not contain 20 attribute heads")
    if summary.get("fine_grained_label_count") != EXPECTED_FINE_GRAINED_LABEL_COUNT:
        raise ValueError("Classification summary has the wrong fine-grained width")
    if summary.get("categories") != list(HIGH_LEVEL_CATEGORIES):
        raise ValueError("Category order is stale")
    if summary.get("attribute_heads") != list(ATTRIBUTE_HEADS):
        raise ValueError("Attribute-head order is stale")
    if summary.get("label_schema") != EXPECTED_HIERARCHY_SCHEMA:
        raise ValueError("Training-derived hierarchy schema is stale")
    if summary.get("model_name") != expected_model_name:
        raise ValueError("Classification summary came from a different checkpoint")
    if summary.get("model_type") != "roberta":
        raise ValueError("Classification summary is not from a RoBERTa encoder")
    if summary.get("pooling_strategy") != "roberta_first_token_dense_tanh":
        raise ValueError("Classification summary uses the wrong pooling strategy")
    assert_exact_policy_split(summary)
    if (
        summary.get("segments") != 3792
        or summary.get("train_segments") != 2352
        or summary.get("eval_segments") != 1440
        or summary.get("train_policies") != 65
        or summary.get("eval_policies") != 50
        or summary.get("train_eval_policy_ids_disjoint") is not True
    ):
        raise ValueError("Classification summary has an unexpected OPP-115 split")

    training_configuration = summary.get("training_configuration", {})
    if not math.isclose(float(training_configuration.get("threshold", float("nan"))), 0.5):
        raise ValueError("The classifier threshold is not the fixed value 0.5")
    if not math.isclose(float(training_configuration.get("attribute_loss_weight", float("nan"))), 1.0):
        raise ValueError("The attribute loss weight is stale")
    if not math.isclose(float(training_configuration.get("classifier_dropout", float("nan"))), 0.15):
        raise ValueError("The classifier dropout is stale")
    if not math.isclose(float(training_configuration.get("positive_weight_power", float("nan"))), 0.5):
        raise ValueError("The positive class-weight power is stale")
    if not math.isclose(float(training_configuration.get("max_positive_weight", float("nan"))), 10.0):
        raise ValueError("The positive class-weight cap is stale")
    if not math.isclose(float(summary.get("target_hierarchical_macro_f1", float("nan"))), TARGET_HIERARCHICAL_MACRO_F1):
        raise ValueError("The reported target score is stale")
    if summary.get("target_reached") not in {True, False}:
        raise ValueError("The target comparison is missing")

    results = summary.get("results", {})
    if results.get("primary_metric") != "hierarchical_macro_f1":
        raise ValueError("Hierarchical macro-F1 is not marked as primary")
    if not math.isclose(float(results.get("threshold", float("nan"))), 0.5):
        raise ValueError("Metric output has the wrong threshold")
    if results.get("evaluated_examples") != summary.get("evaluation_examples_used"):
        raise ValueError("Metric example count does not match evaluated examples")
    assert_unit_interval(results.get("hierarchical_macro_f1"), "hierarchical_macro_f1")
    assert_unit_interval(results.get("balanced_accuracy"), "balanced_accuracy", allow_none=True)

    category = results.get("category", {})
    if not (
        category["primary_metric"] == "macro_f1"
        and category["label_count"] == 10
    ):
        raise ValueError("Category metrics do not cover the 10 category outputs")
    if set(category.get("per_label", {})) != set(HIGH_LEVEL_CATEGORIES):
        raise ValueError("Category per-label metrics are incomplete")
    assert_unit_interval(category.get("macro_f1"), "category macro_f1")

    fine = results.get("fine_grained", {})
    if fine.get("primary_metric") != "fine_grained_macro_f1":
        raise ValueError("Gated fine-grained metrics are missing")
    if fine.get("attribute_head_count") != 20 or fine.get("fine_grained_label_count") != 138:
        raise ValueError("Fine-grained metrics use a stale schema")
    if set(fine.get("per_attribute", {})) != set(ATTRIBUTE_HEADS):
        raise ValueError("Per-attribute metrics do not cover all 20 heads")
    if set(fine.get("per_value", {})) != set(HIERARCHY_SCHEMA.flattened_attribute_labels):
        raise ValueError("Per-value metrics do not cover the frozen 138-value vocabulary")
    assert_unit_interval(fine.get("fine_grained_macro_f1"), "fine-grained macro_f1")

    independent = results.get("independent_attribute_heads", {})
    if independent.get("fine_grained_label_count") != 138:
        raise ValueError("Independent attribute metrics use a stale schema")

    if MODE == "full":
        if summary.get("sampling_applied") is not False:
            raise ValueError("Full mode must use every split example")
        if (
            summary.get("training_examples_used") != 2352
            or summary.get("evaluation_examples_used") != 1440
            or summary.get("training_policies_used") != 65
            or summary.get("evaluation_policies_used") != 50
        ):
            raise ValueError("Full mode did not cover the complete 65/50 split")
    else:
        if summary.get("sampling_applied") is not True:
            raise ValueError("Smoke mode must be explicitly sampled")
        if (
            summary.get("training_examples_used") != 8
            or summary.get("evaluation_examples_used") != 32
            or summary.get("completed_optimizer_steps") != 1
        ):
            raise ValueError("Smoke mode did not complete its fixed one-step check")


def validate_current_classification_summary(summary):
    validate_classification_summary(summary, classifier_checkpoint)


expected_classifier_model = None if MODE == "smoke" else CLASSIFIER_OUTPUT
if stage_is_complete(
    classification_marker_path,
    classification_specification,
    classification_summary_path,
    expected_classifier_model,
    summary_validator=validate_current_classification_summary,
    model_validator=classifier_directory_valid,
):
    print("Reusing completed classification stage")
    classification_summary = json.loads(classification_summary_path.read_text(encoding="utf-8"))
    validate_current_classification_summary(classification_summary)
else:
    classification_command = [
        str(PYTHON_EXECUTABLE),
        str(PROJECT_DIR / "bert_opp115_classification.py"),
        "--data-dir", str(OPP115_DIR),
        "--model-name", classifier_checkpoint,
        "--train-policies", "65",
        "--eval-policies", "50",
        "--threshold", "0.5",
        "--attribute-loss-weight", "1.0",
        "--classifier-dropout", "0.15",
        "--positive-weight-power", "0.5",
        "--max-positive-weight", "10.0",
        "--target-hierarchical-macro-f1", str(TARGET_HIERARCHICAL_MACRO_F1),
        "--summary-file", str(classification_summary_path),
        "--device", TRAINING_DEVICE,
    ]
    if MODE == "smoke":
        classification_command += [
            "--max-length", "64",
            "--batch-size", "4",
            "--max-steps", "1",
            "--max-train-examples", "8",
            "--max-eval-examples", "32",
            "--no-save",
        ]
    else:
        classification_command += [
            "--max-length", "384",
            "--batch-size", "8",
            "--gradient-accumulation-steps", "4",
            "--learning-rate", "2.5e-5",
            "--epochs", "5",
            "--output-dir", str(CLASSIFIER_OUTPUT),
            "--overwrite-output-dir",
        ]
    if not PRETRAINING_READY:
        classification_command += ["--model-revision", BASE_MODEL_REVISION]
    subprocess.run(classification_command, cwd=PROJECT_DIR, check=True)
    if not classification_summary_path.is_file() or (
        expected_classifier_model is not None
        and not classifier_directory_valid(expected_classifier_model)
    ):
        raise RuntimeError("Classification finished without complete saved artifacts")
    classification_summary = json.loads(classification_summary_path.read_text(encoding="utf-8"))
    validate_current_classification_summary(classification_summary)
    write_completion_marker(
        classification_marker_path,
        classification_specification,
        classification_summary_path,
        expected_classifier_model,
    )
print("Classification stage complete.")

## Checks

### 5. Verify the split and display category and fine-grained results

In [ ]:
validate_current_classification_summary(classification_summary)
if pretraining_summary is not None:
    assert pretraining_summary["train_policy_ids"] == classification_summary["train_policy_ids"]
    assert pretraining_summary["eval_policy_ids"] == classification_summary["eval_policy_ids"]
    assert pretraining_summary["evaluated_eval_blocks"] == 0
    assert classification_summary["model_name"] == str(PRETRAIN_OUTPUT)

metrics = classification_summary["results"]
headline = {
    "Run": MODE,
    "Primary metric": "Hierarchical macro-F1",
    "Hierarchical macro-F1": metrics["hierarchical_macro_f1"],
    "Category macro-F1": metrics["category"]["macro_f1"],
    "Gated fine-grained macro-F1": metrics["fine_grained"]["fine_grained_macro_f1"],
    "Target macro-F1": classification_summary["target_hierarchical_macro_f1"],
    "Target reached": classification_summary["target_reached"],
    "Balanced accuracy": metrics["balanced_accuracy"],
    "Threshold": metrics["threshold"],
    "Evaluation examples": metrics["evaluated_examples"],
}
try:
    import pandas as pd
    from IPython.display import display

    display(pd.DataFrame([headline]))
    category_table = pd.DataFrame.from_dict(
        metrics["category"]["per_label"], orient="index"
    )
    category_table.index.name = "OPP-115 category"
    display(category_table[["f1", "balanced_accuracy", "positive_support", "negative_support"]])

    attribute_rows = []
    for head in ATTRIBUTE_HEADS:
        values = metrics["fine_grained"]["per_attribute"][head]
        attribute_rows.append(
            {
                "attribute_head": head,
                "values": values["label_count"],
                "applicable_eval_segments": values["evaluated_examples"],
                "macro_f1": values["macro_f1"],
                "balanced_accuracy": values["balanced_accuracy"],
            }
        )
    display(pd.DataFrame(attribute_rows).set_index("attribute_head"))
except ImportError:
    print(json.dumps(headline, indent=2))
    print(json.dumps(metrics["fine_grained"]["per_attribute"], indent=2))

print("Hierarchy schema SHA-256:", classification_summary["label_schema"]["schema_sha256"])
print("Held-out values absent from the training vocabulary:", classification_summary["held_out_unknown_attribute_values"])
print(
    "Repeated boilerplate evaluation segments also present in training policies:",
    classification_summary["cross_split_duplicate_eval_segments"],
)
if MODE == "smoke":
    print("SMOKE MODE: these values validate execution only; use MODE='full' for final results.")
elif classification_summary["target_reached"]:
    print("Held-out target reached with the observed hierarchical macro-F1.")
else:
    print("Held-out target not reached; report the observed score without test-set tuning.")

## Next steps

1. Choose a Colab GPU runtime and run every cell.
2. Leave `MODE = "full"` for the final experiment; use `"smoke"` only as an
   optional one-step wiring check.
3. Keep a stable `RUN_NAME`. Matching completed stages are reused, while a
   configuration mismatch requires a new name.
4. Use
   `<DRIVE_RUNS_ROOT>/<RUN_NAME>/results/colab_bert_hierarchy_classification_full.json`
   as the durable supervised result artifact.

The workflow uses continued token-masking MLM without next-sentence prediction.
The 50-policy set remains held out from vocabulary construction, pretraining,
and parameter selection. Attribute metrics are reported both independently and
after parent-category gating; the gated result is used in hierarchical
macro-F1. The 0.96 value is an aspirational benchmark, not a guaranteed or
pre-filled result.